# AI TRPG text_model manual test

This notebook tests all currently adapted text models:
- `deepseek`
- `chatgpt`
- `gemini`

It does not require editing `data_UserPreference.json` by hand.


In [ ]:
import json
import os
from pathlib import Path

from openai import OpenAI

from text_model.function_TextModel import (
    collect_stream_text,
    get_mini_reply,
    get_normal_reply,
    get_reply,
    get_selected_text_model_config,
    get_stream_reply,
)


In [ ]:
MODEL_CODES = ["deepseek", "chatgpt", "gemini"]
TMP_PREF_DIR = Path("data") / "_tmp_notebook_prefs"
TMP_PREF_DIR.mkdir(parents=True, exist_ok=True)

messages = [
    {"role": "system", "content": "Reply in concise Chinese."},
    {"role": "user", "content": "请用一句话介绍你自己。"},
]

def make_pref_file(model_code: str) -> Path:
    pref_path = TMP_PREF_DIR / f"pref_{model_code}.json"
    pref_data = {
        "language": "zh-CN",
        "text_model": {"code": model_code},
    }
    pref_path.write_text(json.dumps(pref_data, ensure_ascii=False, indent=2), encoding="utf-8")
    return pref_path

def show_model_info(model_code: str):
    pref_path = make_pref_file(model_code)
    cfg = get_selected_text_model_config(preference_path=pref_path)
    print(f"model_code = {model_code}")
    print("provider =", cfg.dependence)
    print("base_model =", cfg.base_model)
    print("base_url =", cfg.base_url)
    print("mini_supported =", cfg.features["mini_version"].supported)
    print()

def run_and_print(title: str, fn):
    print(title)
    try:
        result = fn()
        print(result)
    except Exception as exc:
        print(type(exc).__name__, exc)
    print()

messages


## Model info check

Run this first to verify model selection and feature flags.


In [ ]:
for code in MODEL_CODES:
    show_model_info(code)


## Gemini model discovery

If Gemini returns 404, run this cell first and use the exact model id returned here in `data_TextModel.yml`.


In [ ]:
client = OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

for model in client.models.list():
    model_id = getattr(model, "id", "")
    if "gemini" in model_id.lower():
        print(model_id)


## Real request tests

Before running the cells below, make sure your `.env` contains valid keys:
- `DEEPSEEK_API_KEY`
- `OPENAI_API_KEY`
- `GEMINI_API_KEY`

These cells continue on errors so you can compare all three providers in one run.


In [ ]:
for code in MODEL_CODES:
    pref_path = make_pref_file(code)
    run_and_print(
        f"===== {code} | normal =====",
        lambda pref_path=pref_path: get_normal_reply(input=messages, preference_path=pref_path, timeout=60),
    )


In [ ]:
for code in MODEL_CODES:
    pref_path = make_pref_file(code)
    run_and_print(
        f"===== {code} | get_reply non-stream =====",
        lambda pref_path=pref_path: get_reply(is_stream=False, input=messages, preference_path=pref_path, timeout=60),
    )


In [ ]:
for code in MODEL_CODES:
    pref_path = make_pref_file(code)
    print(f"===== {code} | stream =====")
    try:
        chunks = []
        for chunk in get_stream_reply(input=messages, preference_path=pref_path, timeout=60):
            print(chunk, end="")
            chunks.append(chunk)
        print("\n")
        print("joined =", collect_stream_text(chunks))
    except Exception as exc:
        print(type(exc).__name__, exc)
    print()


In [ ]:
for code in MODEL_CODES:
    pref_path = make_pref_file(code)
    print(f"===== {code} | get_reply stream =====")
    try:
        chunks = []
        for chunk in get_reply(is_stream=True, input=messages, preference_path=pref_path, timeout=60):
            print(chunk, end="")
            chunks.append(chunk)
        print("\n")
        print("joined =", collect_stream_text(chunks))
    except Exception as exc:
        print(type(exc).__name__, exc)
    print()


In [ ]:
for code in MODEL_CODES:
    pref_path = make_pref_file(code)
    run_and_print(
        f"===== {code} | mini =====",
        lambda pref_path=pref_path: get_mini_reply(input=messages, preference_path=pref_path, timeout=60),
    )


In [ ]:
for path in TMP_PREF_DIR.glob("pref_*.json"):
    path.unlink(missing_ok=True)

print("temporary preference files cleaned")
